# 🔬 AMD Detection Tool - Python/Google Colab Edition

**Version:** 1.5.1  
**Based on:** Rockwell et al. (2021) USGS methodology

This notebook detects **Acid Mine Drainage (AMD)** and **iron sulfate minerals** using satellite imagery from Landsat 8 and Sentinel-2.

---

## 📋 Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Google Earth Engine Authentication](#2-google-earth-engine-authentication)
3. [Define Study Area](#3-define-study-area)
4. [Run AMD Analysis](#4-run-amd-analysis)
5. [Visualize Results](#5-visualize-results)
6. [Export Results](#6-export-results)
7. [Advanced: Custom Analysis](#7-advanced-custom-analysis)

---

## 1. Setup & Installation

First, we install required packages, then upload the AMD detection module.

In [ ]:
# ============================================================================
# STEP 1.1: Install Required Packages (Run once)
# ============================================================================

# Uncomment the following lines if running in Google Colab
# !pip install earthengine-api geemap folium --quiet

print("✅ Packages ready!")

In [ ]:
# ============================================================================
# STEP 1.2: Upload AMD Module (Upload Only)
# ============================================================================

import os
import sys
from google.colab import files

print("📤 Click the button below to upload amd_detection.py:")
print("   (Download it from GitHub first if you don't have it locally)")
print()

uploaded = files.upload()

if 'amd_detection.py' in uploaded:
    print("✅ amd_detection.py uploaded successfully!")
    
    # Save the file
    with open('amd_detection.py', 'wb') as f:
        f.write(uploaded['amd_detection.py'])
    
    # Ensure current directory is in Python path
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    
    print("✅ File saved and Python path configured")
    print("💡 Proceed to Step 2.1 to authenticate and import the module")
    
else:
    print("❌ amd_detection.py not uploaded")
    print("💡 Please run this cell again and select amd_detection.py when prompted")

---

## 2. Google Earth Engine Authentication

You need a Google Earth Engine account. Sign up free at: https://earthengine.google.com/

**Important:** Upload `amd_detection.py` in Step 1.2 BEFORE running this cell!

The module will be imported AFTER GEE authentication in Step 2.1.

In [ ]:
# ============================================================================
# STEP 2.1: Authenticate and Initialize GEE
# ============================================================================

# First, import Google Earth Engine
import ee
import geemap

# Then authenticate (this will open a browser window)
# Follow the prompts and paste the authorization code
print("🔐 Opening authentication window...")
ee.Authenticate()

# Initialize with your department project
print("🚀 Initializing Google Earth Engine...")
ee.Initialize(project="remote-sensing-class-fall-2025")
print("✅ Google Earth Engine initialized successfully!")

# Now we can safely import the AMD detection module
print("\n📦 Importing AMD Detection module...")
import amd_detection as amd

# Print version info
amd.print_version()

---

## 3. Define Study Area

Choose from pre-defined AMD sites or define your own custom area.

In [ ]:
# ============================================================================
# STEP 3.1: List Available Pre-defined Study Areas
# ============================================================================

print("📍 Available Pre-defined Study Areas:")
print("═" * 50)
for name in amd.STUDY_AREAS.keys():
    print(f"  • {name}")
print("═" * 50)

In [ ]:
# ============================================================================
# STEP 3.2: Select Study Area
# ============================================================================

# OPTION A: Use a pre-defined study area
study_area_name = "Iron Mountain, CA"  # Change this to your preferred site
geometry = amd.get_study_area(study_area_name)

# OPTION B: Define custom coordinates (uncomment to use)
# latitude = 40.6722
# longitude = -122.5278
# buffer_meters = 12000
# geometry = ee.Geometry.Point([longitude, latitude]).buffer(buffer_meters)
# study_area_name = "Custom Area"

print(f"✅ Study area selected: {study_area_name}")

In [ ]:
# ============================================================================
# STEP 3.3: Set Date Range
# ============================================================================

# Summer months are recommended to avoid snow/ice interference
start_date = "2023-06-01"
end_date = "2023-09-30"

print(f"📅 Date range: {start_date} to {end_date}")

---

## 3.5 ⚙️ Interactive Settings Panel (Optional)

Adjust detection thresholds interactively before running the analysis.

In [ ]:
# ============================================================================
# STEP 3.5: Interactive Settings Panel
# ============================================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

# Create settings widgets
print("⚙️ AMD Detection Settings Panel")
print("═" * 50)
print("Adjust these values before running analysis (Step 4)")
print()

# Sensor selection
sensor_widget = widgets.Dropdown(
    options=['Landsat 8', 'Landsat 9', 'Sentinel-2'],
    value='Landsat 8',
    description='Sensor:',
    style={'description_width': '150px'}
)

# Iron sulfate threshold
iron_threshold = widgets.FloatSlider(
    value=1.15,
    min=1.0,
    max=2.0,
    step=0.05,
    description='Iron Sulfate Threshold:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

# Ferric Iron 1 threshold
ferric1_threshold = widgets.FloatSlider(
    value=1.40,
    min=1.0,
    max=2.0,
    step=0.05,
    description='Ferric Iron 1 Threshold:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

# NDVI max (vegetation mask)
ndvi_max = widgets.FloatSlider(
    value=0.25,
    min=0.1,
    max=0.5,
    step=0.05,
    description='NDVI Max (veg mask):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

# Green veg threshold
green_veg_threshold = widgets.FloatSlider(
    value=1.5,
    min=1.0,
    max=4.0,
    step=0.1,
    description='Green Veg Threshold:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

# Cloud cover
cloud_cover = widgets.IntSlider(
    value=30,
    min=5,
    max=80,
    step=5,
    description='Max Cloud Cover %:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

# Water quality thresholds
contaminated_water = widgets.FloatSlider(
    value=1.80,
    min=1.0,
    max=3.0,
    step=0.1,
    description='Water Contam. Threshold:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

# Score thresholds
score_moderate = widgets.IntSlider(
    value=3,
    min=1,
    max=5,
    step=1,
    description='Score (Moderate):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

score_severe = widgets.IntSlider(
    value=5,
    min=3,
    max=7,
    step=1,
    description='Score (Severe):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

# Season filter
season_widget = widgets.Dropdown(
    options=['All Year', 'Summer (Jul-Sep)', 'Winter (Dec-Feb)', 
             'Spring (Mar-May)', 'Fall (Sep-Nov)', 'Snow-Free (May-Oct)'],
    value='Summer (Jul-Sep)',
    description='Season Filter:',
    style={'description_width': '150px'}
)

# Display widgets in organized sections
print("📡 SENSOR SETTINGS")
display(sensor_widget, cloud_cover, season_widget)

print("\n🔬 IRON SULFATE DETECTION")
display(iron_threshold, ferric1_threshold)

print("\n🌿 VEGETATION MASKING")
display(ndvi_max, green_veg_threshold)

print("\n💧 WATER QUALITY")
display(contaminated_water, score_moderate, score_severe)

print("\n" + "═" * 50)
print("💡 These values will be used when you run Step 4")

In [ ]:
# ============================================================================
# STEP 3.4: Select Satellite Sensor
# ============================================================================

# Options: "Landsat 8" (30m resolution) or "Sentinel-2" (10m resolution)
sensor = "Landsat 8"

# Maximum cloud cover percentage for filtering images
max_cloud_cover = 30

print(f"🛰️ Sensor: {sensor}")
print(f"☁️ Max cloud cover: {max_cloud_cover}%")

---

## 4. Run AMD Analysis

This section runs the complete AMD detection analysis.

In [ ]:
# ============================================================================
# STEP 4.1: Run Complete Analysis (Uses Settings from Step 3.5)
# ============================================================================

# Build custom settings from the widgets (if Step 3.5 was run)
try:
    custom_settings = amd.DEFAULT_SETTINGS.copy()
    custom_settings["iron_sulfate_threshold"] = iron_threshold.value
    custom_settings["ferric_iron1_threshold"] = ferric1_threshold.value
    custom_settings["ndvi_max"] = ndvi_max.value
    custom_settings["green_veg_threshold"] = green_veg_threshold.value
    custom_settings["contaminated_water_threshold"] = contaminated_water.value
    custom_settings["score_threshold_moderate"] = score_moderate.value
    custom_settings["score_threshold_severe"] = score_severe.value
    custom_settings["season_filter"] = season_widget.value
    sensor = sensor_widget.value
    max_cloud_cover = cloud_cover.value
    print("✅ Using custom settings from Settings Panel (Step 3.5)")
except NameError:
    # Widgets not defined, use defaults
    custom_settings = None
    print("ℹ️ Using default settings (Step 3.5 was not run)")

print()
print("🔄 Running AMD analysis...")
print(f"   Study Area: {study_area_name}")
print(f"   Date Range: {start_date} to {end_date}")
print(f"   Sensor: {sensor}")
print(f"   Max Cloud Cover: {max_cloud_cover}%")
print()

# Run the analysis
results = amd.analyze_region(
    geometry=geometry,
    start_date=start_date,
    end_date=end_date,
    sensor=sensor,
    settings=custom_settings,
    cloud_cover_max=max_cloud_cover
)

print("✅ Analysis complete!")
print()
print("📊 Results contain:")
print("   • composite: Satellite image with 19 spectral indices")
print("   • land_classification: 19-class AMD mineral map")
print("   • water_quality: Water contamination assessment (0-2)")
print("   • iron_sulfate: Iron sulfate index layer")
print()
print("📌 Proceed to Step 5 to visualize results with legend")

In [ ]:
# ============================================================================
# STEP 4.2: View Classification Legend
# ============================================================================

amd.print_class_legend()

---

## 5. Visualize Results

Create interactive maps to explore the AMD detection results.

In [ ]:
# ============================================================================
# STEP 5.1: Create Interactive Map with ALL Layers and Legend
# ============================================================================

# Create map centered on study area
Map = amd.create_map(zoom=12)

# Add all result layers (same structure as JavaScript version)
# Now includes ALL diagnostic layers:
#   PRIMARY LAYERS (visible):
#     - 🏔️ Land AMD Classification (19 classes)
#     - 🌊 Water Quality Classification
#   REFERENCE LAYERS (hidden):
#     - 📷 True Color (RGB)
#     - 📷 False Color (NIR-R-G)
#   SPECTRAL INDEX LAYERS (hidden):
#     - 🔬 Iron Sulfate Index
#     - 🔬 Ferric Iron 1 & 2
#     - 🔬 Ferrous Iron
#     - 🔬 Clay-Sulfate-Mica
#     - 🔬 NDVI (Vegetation)
#     - 🔬 MNDWI (Water)
#   WATER QUALITY DIAGNOSTICS (hidden):
#     - 📊 Water Score (0-7)
#     - 🔬 NIR Anomaly (Water)
#     - 🔬 Turbidity Ratio
#     - 🔬 Iron Water Index
#     - 🔬 Yellow Index
#     - 🔬 Brightness

Map = amd.add_results_to_map(Map, results, geometry)

# Add legends to the map
Map = amd.add_legend_to_map(Map)
Map = amd.add_water_legend_to_map(Map)

# Center map on study area
Map.centerObject(geometry, 12)

# Display the map
print("🗺️ Interactive Map with Legend")
print("═" * 50)
print("📌 Use the LAYER CONTROL (top right) to toggle layers")
print("📌 LEGENDS are shown at bottom corners")
print()
print("COLOR GUIDE:")
print("   🔴 RED/CRIMSON = Iron Sulfate minerals (AMD)")
print("   🟣 MAGENTA = Major Ferric Iron")
print("   🟡 YELLOW/ORANGE = Clay minerals")
print("   🟢 GREEN = Vegetation")
print("   🔵 BLUE water = Clean")
print("   🟠 ORANGE/RED water = Contaminated")
print("═" * 50)
Map

In [ ]:
# ============================================================================
# STEP 5.2: View Iron Sulfate Index Only
# ============================================================================

# Create a new map for iron sulfate visualization
iron_map = amd.create_map(zoom=12)

# Add iron sulfate with diagnostic visualization
iron_map.addLayer(
    results['iron_sulfate'], 
    amd.VIS_IRON_SULFATE, 
    '🔬 Iron Sulfate Index'
)

# Add true color for reference
iron_map.addLayer(
    results['composite'],
    amd.VIS_TRUE_COLOR,
    '📷 True Color',
    False
)

iron_map.centerObject(geometry, 12)

print("🔬 Iron Sulfate Index Map (Rockwell et al. 2021)")
print("   Formula: (Blue + Red) / Coastal")
print("   Threshold: > 1.15 indicates iron sulfate minerals")
print()
print("   Color Scale:")
print("   Cyan   = Near threshold (1.15)")
print("   Yellow = Moderate (2.0)")
print("   Orange = High (3.0)")
print("   Red    = Very High (4.0+)")

iron_map

---

## 6. Export Results

Export classification results to Google Drive for further analysis in GIS software.

In [ ]:
# ============================================================================
# STEP 6.1: Export Land Classification to Google Drive
# ============================================================================

# Create export task
export_task = amd.export_to_drive(
    image=results['land_classification'],
    description=f"AMD_Classification_{study_area_name.replace(' ', '_').replace(',', '')}",
    folder="AMD_Detection_Results",
    region=geometry,
    scale=30  # 30m for Landsat, use 10 for Sentinel-2
)

# Start the export (uncomment to run)
# export_task.start()
# print("✅ Export started! Check your Google Drive in a few minutes.")

print("💡 To start the export, uncomment the lines above and run this cell.")

In [ ]:
# ============================================================================
# STEP 6.2: Export Multiple Layers (Batch Export)
# ============================================================================

# Define exports
exports = [
    ('land_classification', 'AMD_Land_Classification'),
    ('iron_sulfate', 'AMD_Iron_Sulfate_Index'),
]

# Create tasks
tasks = []
for layer_name, export_name in exports:
    task = amd.export_to_drive(
        image=results[layer_name],
        description=export_name,
        folder="AMD_Detection_Results",
        region=geometry,
        scale=30
    )
    tasks.append((export_name, task))
    print(f"📦 Export task created: {export_name}")

print()
print("💡 To start all exports, run:")
print("   for name, task in tasks:")
print("       task.start()")
print("       print(f'Started: {name}')")

---

## 7. Advanced: Custom Analysis

For advanced users who want to customize thresholds or analyze specific indices.

In [ ]:
# ============================================================================
# STEP 7.1: View Default Settings
# ============================================================================

print("⚙️ Default Detection Settings:")
print("═" * 50)
for key, value in amd.DEFAULT_SETTINGS.items():
    print(f"  {key}: {value}")
print("═" * 50)

In [ ]:
# ============================================================================
# STEP 7.2: Run Analysis with Custom Settings
# ============================================================================

# Create custom settings (copy defaults and modify)
custom_settings = amd.DEFAULT_SETTINGS.copy()

# Example: Adjust iron sulfate threshold for more/less sensitive detection
custom_settings["iron_sulfate_threshold"] = 1.10  # Lower = more sensitive
custom_settings["ndvi_max"] = 0.20  # Stricter vegetation mask

# Run with custom settings
custom_results = amd.analyze_region(
    geometry=geometry,
    start_date=start_date,
    end_date=end_date,
    sensor=sensor,
    settings=custom_settings
)

print("✅ Custom analysis complete!")

In [ ]:
# ============================================================================
# STEP 7.3: Compare Results (Default vs Custom)
# ============================================================================

compare_map = amd.create_map(zoom=12)

# Add default results
compare_map.addLayer(
    results['land_classification'],
    {'min': 0, 'max': 19, 'palette': amd.CLASSIFICATION_PALETTE},
    'Default Classification'
)

# Add custom results
compare_map.addLayer(
    custom_results['land_classification'],
    {'min': 0, 'max': 19, 'palette': amd.CLASSIFICATION_PALETTE},
    'Custom Classification (Sensitive)',
    False  # Hidden by default
)

compare_map.centerObject(geometry, 12)
print("💡 Toggle layers in the map controls to compare results.")
compare_map

In [ ]:
# ============================================================================
# STEP 7.4: Enable Adaptive Thresholds (Paper Section 3.3)
# ============================================================================

# Adaptive thresholds are recommended for heterogeneous terrain (mountains)
# They calculate: threshold = mean + (N × std_dev)

adaptive_settings = amd.DEFAULT_SETTINGS.copy()
adaptive_settings["use_adaptive_thresholds"] = True
adaptive_settings["iron_std_mult"] = 2.0  # 2 standard deviations above mean
adaptive_settings["ferric_std_mult"] = 1.5

print("📊 Adaptive thresholds enabled!")
print("   Iron multiplier: 2.0σ")
print("   Ferric multiplier: 1.5σ")
print()
print("💡 Use this for mountainous/heterogeneous terrain.")

---

## 📚 Reference

This tool is based on:

> **Rockwell, B.W., 2021**, Mapping acid-generating minerals, acidic drainage, iron sulfate minerals, and other mineral groups using Landsat 8 Operational Land Imager data, San Juan Mountains, Colorado, and Four Corners Region. *U.S. Geological Survey Scientific Investigations Map 3466*.

---

## 🛠️ Troubleshooting

**Q: Authentication failed?**
- Make sure you have a GEE account: https://earthengine.google.com/
- Try running `ee.Authenticate()` manually

**Q: No data found?**
- Check your date range - summer months work best
- Reduce cloud cover threshold
- Verify your geometry is correct

**Q: Results look wrong?**
- Compare with true color imagery
- Check the classification legend
- Try adjusting thresholds

---

**Happy AMD Detection! 🔬**